# Notebook 03 — Análisis exploratorio de datos (EDA)

**Objetivo:** Explorar el dataset procesado mediante análisis univariado, bivariado y multivariado, respondiendo las preguntas definidas por el grupo.

**Preguntas guía:**
1. ¿Fumar es el factor que más influye en los costos médicos?
2. ¿A mayor IMC y edad, mayores costos? ¿Se potencian entre sí?
3. ¿Existen grupos diferenciados de pacientes según su perfil de riesgo?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
df = pd.read_csv('../data/processed/reporte_clinica_clean.csv')
print(f'Dataset cargado. Dimensiones: {df.shape}')
df.head()

---
## Análisis univariado
### Visualización 1 — Distribución de costos médicos (`charges`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de charges')
axes[0].set_xlabel('Costo médico (USD)')
axes[0].set_ylabel('Frecuencia')

axes[1].boxplot(df['charges'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot de charges')
axes[1].set_ylabel('Costo médico (USD)')

plt.suptitle('Variable objetivo: charges', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/viz_01_charges.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Media: {df["charges"].mean():.2f} | Mediana: {df["charges"].median():.2f}')
print(f'Mínimo: {df["charges"].min():.2f} | Máximo: {df["charges"].max():.2f}')

**Interpretación:** La distribución de `charges` presenta una asimetría positiva marcada: la mayoría de los pacientes tiene costos bajos (mediana ~9.300 USD), pero existe una cola derecha con casos de costos muy elevados (hasta ~63.000 USD). Esta bimodalidad sugiere la existencia de al menos dos grupos de pacientes con perfiles de riesgo diferentes, lo cual es coherente con la pregunta 3 del proyecto.

### Visualización 2 — Distribución de fumadores y no fumadores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

conteo = df['smoker'].value_counts()
axes[0].bar(conteo.index, conteo.values, color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_title('Cantidad de fumadores y no fumadores')
axes[0].set_xlabel('Fumador')
axes[0].set_ylabel('Cantidad')
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(conteo.values, labels=conteo.index, autopct='%1.1f%%',
            colors=['steelblue', 'coral'], startangle=90)
axes[1].set_title('Proporción de fumadores')

plt.suptitle('Variable: smoker', fontsize=13)
plt.tight_layout()
plt.savefig('../reports/viz_02_smoker.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretación:** Aproximadamente el 20% de los pacientes del dataset son fumadores, mientras el 80% no fuma. Esta proporción implica que el grupo de fumadores es minoritario pero relevante: si el tabaquismo tiene un impacto fuerte sobre los costos, ese 20% podría explicar una parte desproporcionada de la varianza en `charges`, lo cual motivará el análisis bivariado siguiente.

---
## Análisis bivariado
### Visualización 3 — Costos médicos según condición de fumador (Pregunta 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df.boxplot(column='charges', by='smoker', ax=axes[0],
           patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.5))
axes[0].set_title('Distribución de charges por condición de fumador')
axes[0].set_xlabel('Fumador')
axes[0].set_ylabel('Costo médico (USD)')
plt.sca(axes[0])
plt.title('charges por smoker')

medias = df.groupby('smoker')['charges'].mean()
axes[1].bar(medias.index, medias.values, color=['steelblue', 'coral'], edgecolor='white')
axes[1].set_title('Costo promedio según condición de fumador')
axes[1].set_xlabel('Fumador')
axes[1].set_ylabel('Costo medio (USD)')
for i, (k, v) in enumerate(medias.items()):
    axes[1].text(i, v + 200, f'${v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/viz_03_charges_smoker.png', dpi=150, bbox_inches='tight')
plt.show()

print('Costos medios por grupo:')
print(df.groupby('smoker')['charges'].agg(['mean','median','std']).round(2))

**Interpretación (Pregunta 1):** Los pacientes fumadores tienen un costo medio de aproximadamente 32.000 USD frente a 8.400 USD de los no fumadores, lo que representa una diferencia de casi 4 veces. La mediana de los fumadores también supera ampliamente al percentil 75 de los no fumadores. Esta evidencia permite concluir preliminarmente que el tabaquismo es el factor de mayor impacto individual sobre los costos médicos en este dataset. El análisis multivariado y el PCA permitirán confirmar si este patrón se mantiene al controlar otras variables.

### Visualización 4 — Relación entre IMC, edad y costos (Pregunta 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc1 = axes[0].scatter(df['bmi'], df['charges'],
                      c=df['smoker'].map({'yes': 1, 'no': 0}),
                      cmap='coolwarm', alpha=0.5, s=20)
axes[0].set_title('IMC vs Costos médicos (color: fumador)')
axes[0].set_xlabel('IMC (bmi)')
axes[0].set_ylabel('Costo médico (USD)')
plt.colorbar(sc1, ax=axes[0], label='Fumador (1=sí)')

sc2 = axes[1].scatter(df['age'], df['charges'],
                      c=df['smoker'].map({'yes': 1, 'no': 0}),
                      cmap='coolwarm', alpha=0.5, s=20)
axes[1].set_title('Edad vs Costos médicos (color: fumador)')
axes[1].set_xlabel('Edad (age)')
axes[1].set_ylabel('Costo médico (USD)')
plt.colorbar(sc2, ax=axes[1], label='Fumador (1=sí)')

plt.tight_layout()
plt.savefig('../reports/viz_04_scatter_bmi_age.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlaciones con charges:')
print(df[['age','bmi','children','charges']].corr()['charges'].round(3))

**Interpretación (Pregunta 2):** Los scatterplots revelan una estructura clara: los pacientes fumadores (rojo) forman una banda de costos notablemente más elevados que los no fumadores (azul) independientemente del IMC o la edad. Sin embargo, dentro del grupo de fumadores se observa que tanto la edad como el IMC elevado acentúan los costos: los puntos rojos en la parte superior corresponden a pacientes mayores o con mayor IMC. La correlación de `age` con `charges` es de 0.30 y la de `bmi` de 0.20, ambas positivas pero moderadas, lo que confirma que estas variables contribuyen pero de forma secundaria respecto al tabaquismo.

---
## Análisis multivariado
### Visualización 5 — Mapa de calor de correlaciones + perfil de grupos

In [ ]:
df_num = df.copy()
df_num['smoker_num'] = df['smoker'].map({'yes': 1, 'no': 0})
df_num['sex_num'] = df['sex'].map({'male': 1, 'female': 0})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr = df_num[['age','bmi','children','smoker_num','sex_num','charges']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[0], linewidths=0.5)
axes[0].set_title('Mapa de correlaciones')

# Perfil multivariado: costo medio por región y smoker
pivot = df.groupby(['region', 'smoker'])['charges'].mean().unstack()
pivot.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'], edgecolor='white')
axes[1].set_title('Costo medio por región y condición de fumador')
axes[1].set_xlabel('Región')
axes[1].set_ylabel('Costo medio (USD)')
axes[1].legend(title='Fumador')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/viz_05_multivariado.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretación (Pregunta 3):** El mapa de correlaciones confirma que `smoker_num` es la variable con mayor correlación con `charges` (r≈0.79), muy por encima de `age` (r≈0.30) y `bmi` (r≈0.20). Las demás variables muestran correlaciones débiles con la variable objetivo. El gráfico de barras por región revela que el patrón se mantiene consistente en todas las regiones: en todos los casos, los fumadores tienen costos entre 3 y 4 veces más altos que los no fumadores. Esto indica que el efecto del tabaquismo no está mediado por la región geográfica. Estos patrones estructurados sugieren la existencia de grupos diferenciados que serán explorados formalmente mediante PCA en el notebook siguiente.